## Спецификация модели

Пусть

- $y$ - бинарная зависимая переменная
- $(x_1,\ldots,x_k)$ - объясняющие переменные или регрессоры

Обозначим

$$
\begin{aligned}
	x^\top&=\begin{pmatrix} 1 & x_1 & \cdots & x_k\end{pmatrix} &
	\beta^\top&=\begin{pmatrix} \beta_0 & \beta_1 & \cdots & \beta_k\end{pmatrix} &
\end{aligned}
$$

Тогда

$$
	x^\top\beta=\beta_0+\beta_1x_1+\cdots+\beta_kx_k
$$

Базовые модели регрессии бинарной зависимой переменной

$$
\begin{aligned}
	logit:&\  Prob(y=1)=\Lambda\left(x^\top\beta\right) \\
	probit:&\  Prob(y=1)=\Phi\left(x^\top\beta\right)
\end{aligned}
$$

где 

$$
\begin{aligned}
	\Lambda(u)&=\frac{\exp(u)}{1+\exp(u)}=\frac{1}{1+\exp(-u)} & 
	\Phi(u)&=\frac{1}{\sqrt{2\pi}}\int_{-\infty}^u \exp\left(-t^2/2\right)dt
\end{aligned}
$$

Альтернативные спецификации

$$
\begin{aligned}
	logit(Prob(y=1))&= x^\top\beta \\
	probit(Prob(y=1))&= x^\top\beta
\end{aligned}
$$

где

$$
\begin{aligned}
	logit(p)& =\Lambda^{-1}(p)=\log\left(\frac{p}{1-p}\right) &
	probit(p)&=\Phi^{-1}(p)
\end{aligned}
$$

In [ ]:

import numpy as np
import pandas as pd

from statsmodels.formula.api import logit, probit

from statsmodels.iolib.summary2 import summary_params, summary_col # вывод результатов тестирования

# критические значения
from scipy.stats import norm, chi2

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

import matplotlib.pyplot as plt

from IPython.display import Markdown, display

# загружен датасеты в виде словаря
path_to_datasets = '../../datasets/econometrica-2/'
df_dict = {'loanapp': pd.read_csv(path_to_datasets+'loanapp.csv'), 
			'TableF5-1':pd.read_csv(path_to_datasets+'TableF5-1.csv'), 
			'SwissLabour': pd.read_csv(path_to_datasets+'SwissLabour.csv')}

## Оценивание и интерпретация коэффициентов

In [ ]:
df_names = ['loanapp', 'loanapp', 'TableF5-1', 'TableF5-1', 'SwissLabour', 'SwissLabour']

endog_var = ['approve', 'approve', 'LFP', 'LFP', 'participation', 'participation']

exog_vars =['mortno+unem+dep+male+married+yjob+self', 
			'appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'WA+WE+CIT+UN+np.log(FAMINC)',
			'income+age+education+youngkids+oldkids', 
			'np.log(income)+age+I(age**2)+education+youngkids+oldkids']

mod_name = ['Approve equation №1', 'Approve equation №2', 'Labour force equation №1', 'Labour force equation №2', 'Swiss labour force equation №1', 'Swiss labour force equation №2']

mod_type = ['logit', 'probit', 'logit', 'probit', 'logit', 'probit']

digits = 3

for i, df_name in enumerate(df_names):
	if mod_type[i] == 'logit':
		mod = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	else:
		mod = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])

	res = mod.fit(disp=False)

	display(Markdown(f'### {mod_name[i]} ({mod_type[i]})'))
	display(Markdown(f'''Для датасет `{df_name}` оцените __{mod_type[i]}__ регрессию `{endog_var[i]}` на `{exog_vars[i]}`
	и укажите её коэффициенты. __Ответ округлите до {digits}-х десятичных знаков__ '''))

	df_coefs = pd.DataFrame({'Коэффициент':res.params.index, 'Оценка':res.params.round(digits)})
	display(Markdown(f'''Ответ:\n'''))
	display(Markdown(df_coefs.to_markdown(index=False)))
	# print(df_coefs.to_markdown(index=False))
	display(Markdown(f'''__Дайте интерпретацию коэффициентам модели__'''))

## z-тест

In [ ]:
df_names = ['loanapp', 'loanapp', 'TableF5-1', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'approve', 'LFP', 'LFP', 'participation']

exog_vars =['mortno+unem+dep+male+married+yjob+self', 
			'appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'WA+WE+CIT+UN+np.log(FAMINC)',
			'income+I(income**2)+age+I(age**2)+education+youngkids+oldkids+foreign',
			'np.log(income)+I(np.log(income)**2)+age+I(age**2)+education+youngkids+oldkids+foreign']

mod_name = ['Approve equation №1', 'Approve equation №2', 'Labour force equation №1', 'Labour force equation №2', 'Swiss labour force equation №1', 'Swiss labour force equation №2']

mod_type = ['logit', 'probit', 'logit', 'probit', 'logit', 'probit']

digits = 3

sign_level = [0.05, 0.05, 0.01, 0.05, 0.10, 0.01]

for i, df_name in enumerate(df_names):
	if mod_type[i] == 'logit':
		mod = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	else:
		mod = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])

	res = mod.fit(disp=False)

	display(Markdown(f'### {mod_name[i]} ({mod_type[i]})'))
	display(Markdown(f'''Для датасет `{df_name}` оцените __{mod_type[i]}__ регрессию `{endog_var[i]}` на `{exog_vars[i]}`
	и тестируйте значимость её коэффициентов. Уровень значимости {sign_level[i]*100}%. 
	Для каждого коэффициента укажите\n - его тестовую статистику,\n - P-значение\n - критическое значение\n - седлайте вывод.\n\n  __Ответ округлите до {digits}-х десятичных знаков__ '''))

	res_z_test = pd.DataFrame({'Коэффициент':res.params.index, 
								'Тест стат':res.tvalues.round(digits),
								'P-значение':res.pvalues.round(digits),
								'Вывод': res.pvalues.apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим') })
	display(Markdown(f'''Ответ:\n'''))
	display(Markdown(res_z_test.to_markdown(index=False)))
	# print(df_coefs.to_markdown(index=False))

	crit_val = norm.isf(q=sign_level[i]/2).round(digits)
	display(Markdown(f'''Критическое значение: {crit_val}'''))

## LR-тест: значимость регрессии

Тестируем гипотезу

$$
	H_0:\beta_1=\cdots=\beta_k=0
$$

Тестовая статистика

$$
	LR=2\left(\log L-\log L_0\right)
$$

In [ ]:
df_names = ['loanapp', 'loanapp', 'TableF5-1', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'approve', 'LFP', 'LFP', 'participation']

exog_vars =['appinc+unem+male+yjob+self', 
			'appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'K618+CIT+UN',
			'WA+I(WA**2)+K618+CIT+UN',
			'income+I(income**2)+education']

mod_name = ['Approve equation №1', 'Approve equation №2', 'Labour force equation №1', 'Labour force equation №2', 'Swiss labour force equation №1']

mod_type = ['probit', 'logit', 'logit', 'probit', 'logit']

digits = 3

sign_level = [0.10, 0.05, 0.05, 0.01, 0.10]

for i, df_name in enumerate(df_names):
	if mod_type[i] == 'logit':
		mod = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	else:
		mod = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])

	res = mod.fit(disp=False)

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'''Для датасет `{df_name}` оцените __{mod_type[i]}__ регрессию `{endog_var[i]}` на `{exog_vars[i]}`.
	Тестируйте значимость регрессии. Уровень значимости __{sign_level[i]*100}%__. 
	Найдите\n - значение тестовой статистики,\n - её P-значение,\n - критическое значение\n - сделайте вывод.\n\n __Ответ округлите до {digits}-х десятичных знаков__. Какие коэффициенты значимы?'''))

	test_stat = res.llr
	p_val = res.llr_pvalue
	crit_val = chi2.isf(sign_level[i]/2, df=res.df_model)
	is_significant = 'значима' if p_val<sign_level[i] else 'незначима' 

	res_lr_test_overall = pd.DataFrame({'Статистика': [round(test_stat, digits)], 
										'P-значение': [round(p_val, digits)],
										'Критическое': [crit_val.round(digits)], 
										'Регрессия': [is_significant]})

	res_z_test = pd.DataFrame({'Коэффициент':res.params.index, 
								'Тест стат':res.tvalues.round(digits),
								'P-значение':res.pvalues.round(digits),
								'Вывод': res.pvalues.apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим') })

	display(Markdown(f'''Ответ:\n'''))
	display(Markdown(res_lr_test_overall.to_markdown(index=False)))

	display(Markdown(res_z_test.to_markdown(index=False)))

## Тест Вальда: совместная значимость

In [ ]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =['appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'income+I(income**2)+age+I(age**2)+education+youngkids+oldkids+foreign']

mod_name = ['Approve equation №1', 'Labour force equation #1', 'Swiss labour force equation №1']

mod_type = ['probit', 'logit', 'logit']

digits = 3

sign_level = [[0.01, 0.05, 0.01], [0.01, 0.10, 0.10], [0.10, 0.05, 0.10]]

joint_sign = [ ['appinc, appinc**2', 'male, yjob, self', 'male, yjob, self, unem, married'], 
				['WA, WA**2', 'K618, CIT, UN', 'K618, CIT, UN, FAMINC'], 
				['income, income**2', 'age, age**2', 'youngkids, oldkids'] ]

hypotheses = [ ['appinc=0, I(appinc ** 2)=0', 'male=0, yjob=0, self=0', 'male=0, yjob=0, self=0, unem=0, married=0'], 
				['WA=0, I(WA ** 2)=0', 'K618=0, CIT=0, UN=0', 'K618=0, CIT=0, UN=0, np.log(FAMINC)=0'],
				['income=0, I(income ** 2)=0', 'age=0, I(age ** 2)=0', 'youngkids=0, oldkids=0'] ]

for i, df_name in enumerate(df_names):
	if mod_type[i] == 'logit':
		mod = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	else:
		mod = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])

	res = mod.fit(disp=False)

	display(Markdown(f'### {mod_name[i]} ({mod_type[i]})'))
	display(Markdown(f'''Для датасет `{df_name}` оцените __mod_type[i]__ регрессию `{endog_var[i]}` на `{exog_vars[i]}`
	Тестируйте гипотезы ниже. Для каждой гипотезы\n - дайте интерпретацию,\n - найдите тестовую статистику,\n - её P-значение,\n - критическое значение\n - сделайте вывод.\n\n __Ответ округлите до {digits}-х десятичных знаков__ '''))

	for j, hypothesis in enumerate(hypotheses[i]):
		display(Markdown(f'#### Гипотеза №{j+1}'))
		display(Markdown(f'Тестируйте совместную значимость `{joint_sign[i][j]}`. Уровень значимость __{sign_level[i][j]*100}%__'))

		res_wald_test = res.wald_test(hypothesis, use_f=False)

		test_stat = res_wald_test.statistic
		p_val = res_wald_test.pvalue
		crit_val = chi2.isf(sign_level[i][j], df = res_wald_test.df_denom)
		is_significant = 'значимо' if p_val<sign_level[i][j] else 'незначимо'

		res_wald_test_joint = pd.DataFrame({'Статистика': [round(test_stat, digits)], 
										'P-значение': [round(p_val, digits)],
										'Критическое': [crit_val.round(digits)], 
										'Совместное влияние': [is_significant]})

		display(Markdown(f'''Ответ:\n'''))
		display(Markdown(res_wald_test_joint.to_markdown(index=False)))